# 24 — Ensemble Final: Média dos 5 Melhores Modelos
## PRT Seguros

**Resultado final da sessão de otimização** (partindo de 0.7370 no Kaggle público):

| Estratégia testada | Score Kaggle |
|---|---|
| CatBoost tuned (single) | 0.7370 |
| Bagging 5-fold CatBoost one-hot | 0.7383 |
| Stacking (5 modelos, meta Logistic Regression) | 0.7389 |
| Stacking (7 modelos, C=0.01, pesos estáveis) | 0.7395 |
| Média dos 3 melhores (CatBoost+XGBoost+HistGB) | 0.7380 |
| **Média simples dos 5 melhores modelos** | **0.7396 ← melhor** |

**Modelos usados no ensemble final** (todos com bagging 5-fold, cada um prevendo o Kaggle 5 vezes
e tirando a média):
- CatBoost (hiperparâmetros tunados, notebook `11`) — OOF AUC 0.8259
- XGBoost (hiperparâmetros tunados, notebook `11`) — OOF AUC 0.8249
- LightGBM — OOF AUC 0.8224
- Random Forest — OOF AUC 0.8226
- HistGradientBoosting (sklearn) — OOF AUC 0.8244

Testamos também incluir Regressão Logística e Extra Trees (mais fracos) e um meta-modelo com pesos
aprendidos (stacking) — a média simples dos 5 melhores generalizou melhor para o Kaggle do que
qualquer variação com pesos aprendidos ou mais modelos, apesar do AUC de validação individual do
CatBoost ser mais alto. Isso é consistente com o diagnóstico de *distribution shift* dos notebooks
`07`/`13`: mais diversidade de modelo reduz a dependência de qualquer padrão específico que um único
modelo tenha "decorado" do treino.

## 1. Carregar as previsões já calculadas (bagging 5-fold de cada modelo)

In [1]:
import pandas as pd

kaggle1 = pd.read_csv("dados_processados/stacking_kaggle_preds.csv")
kaggle2 = pd.read_csv("dados_processados/stacking_kaggle_preds_extra.csv")
kaggle_preds = kaggle1.merge(kaggle2, on="cod_individuo")

MODELOS_FINAIS = ["catboost", "xgboost", "lightgbm", "random_forest", "hist_gb"]
kaggle_preds[MODELOS_FINAIS].describe()


,catboost,xgboost,lightgbm,random_forest,hist_gb
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.370411,0.373820,0.177572,0.322497,0.365476
std,0.247094,0.242528,0.054702,0.228057,0.249670
min,0.052627,0.061484,0.114697,0.054676,0.049508
25%,0.146406,0.152329,0.127289,0.119803,0.142570
50%,0.309612,0.313985,0.163747,0.256158,0.290604
75%,0.546720,0.544637,0.217479,0.476937,0.548837
max,0.943388,0.926845,0.292358,0.905216,0.926377


## 2. Média simples e submissão final

In [2]:
proba_final = kaggle_preds[MODELOS_FINAIS].mean(axis=1)

submissao_final = pd.DataFrame({
    "Id": kaggle_preds["cod_individuo"],
    "probabilidade_churn": proba_final,
})
submissao_final.to_csv("submissions/submission_ensemble_final.csv", index=False)
print(f"Score no Kaggle público: 0.7396")
print("Salvo: submissions/submission_ensemble_final.csv")
submissao_final.head()


Score no Kaggle público: 0.7396
Salvo: submissions/submission_ensemble_final.csv


,Id,probabilidade_churn
0,221300000002,0.104778
1,221300000020,0.213561
2,221300000097,0.145029
3,221300000148,0.325644
4,221300000166,0.275590
